# Grocery Data Scraper
Scrapes grocery product data from SM Markets (smmarkets.ph).

**Output fields:** `item_category`, `item_name`, `brand`, `unit_size`, `price_php`, `store`, `date_scraped`

In [1]:
# Install dependencies if needed
# !pip install selenium tqdm pandas

In [2]:
import runpy
runpy.run_path('grocery_scraper.py', run_name='__main__')

Fetching product list from sitemap...
Found 10052 products in sitemap.
Categorized: 8866 / 10052 from name matching.
Fetching prices with 8 workers...


Fetching prices: 100%|██████████| 10052/10052 [12:38<00:00, 13.26it/s]



Done! 10052 products saved to: results/grocery_data.csv
Unique categories: 63
Products with price: 0
                      item_category                                     item_name    brand unit_size price_php       store date_scraped
0  Babies & Kids - Baby Food & Care                    Aveeno Baby Lotion | 532ml   Aveeno     532ml      None  SM Markets   2026-04-25
1  Babies & Kids - Baby Food & Care            Aveeno Baby Wash & Shampoo | 532ml   Aveeno     532ml      None  SM Markets   2026-04-25
2  Babies & Kids - Baby Food & Care     Aveeno Baby Wash Clnsing Theraphy | 236ml   Aveeno     236ml      None  SM Markets   2026-04-25
3  Babies & Kids - Baby Food & Care     Babyflo Baby Cologne Pink Fantasy | 100ml  Babyflo     100ml      None  SM Markets   2026-04-25
4  Babies & Kids - Baby Food & Care      Babyflo Baby Cologne Purple Rain | 100ml  Babyflo     100ml      None  SM Markets   2026-04-25
5  Babies & Kids - Baby Food & Care  Babyflo Baby Conditioner Shampoo Red | 200ml 

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'grocery_scraper.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromlist=(), level=0)>,
  'abs': <function

In [3]:
# Optional: load and preview results after scraping
import pandas as pd

df = pd.read_csv('results/grocery_data.csv')
print(f'Total products: {len(df)}')
print(f'Stores: {df["store"].unique()}')
print(f'Categories: {df["item_category"].nunique()}')
df.head(20)

Total products: 10052
Stores: ['SM Markets']
Categories: 63


,item_category,item_name,brand,unit_size,price_php,store,date_scraped
0,Babies & Kids - Baby Food & Care,Aveeno Baby Lotion | 532ml,Aveeno,532ml,NaN,SM Markets,2026-04-25
1,Babies & Kids - Baby Food & Care,Aveeno Baby Wash & Shampoo | 532ml,Aveeno,532ml,NaN,SM Markets,2026-04-25
2,Babies & Kids - Baby Food & Care,Aveeno Baby Wash Clnsing Theraphy | 236ml,Aveeno,236ml,NaN,SM Markets,2026-04-25
3,Babies & Kids - Baby Food & Care,Babyflo Baby Cologne Pink Fantasy | 100ml,Babyflo,100ml,NaN,SM Markets,2026-04-25
4,Babies & Kids - Baby Food & Care,Babyflo Baby Cologne Purple Rain | 100ml,Babyflo,100ml,NaN,SM Markets,2026-04-25
5,Babies & Kids - Baby Food & Care,Babyflo Baby Conditioner Shampoo Red | 200ml,Babyflo,200ml,NaN,SM Markets,2026-04-25
6,Babies & Kids - Baby Food & Care,Babyflo Baby Natural Shampoo green | 200ml,Babyflo,200ml,NaN,SM Markets,2026-04-25
7,Babies & Kids - Baby Food & Care,Babyflo Baby Powder Blushy Pink | 100g,Babyflo,100g,NaN,SM Markets,2026-04-25
8,Babies & Kids - Baby Food & Care,Babyflo Baby Powder Blushy Pink | 200g,Babyflo,200g,NaN,SM Markets,2026-04-25
9,Babies & Kids - Baby Food & Care,Babyflo Baby Powder Charming Blue | 200g,Babyflo,200g,NaN,SM Markets,2026-04-25


In [4]:
import re

# Recategorize items based on name patterns

df = pd.read_csv('results/grocery_data.csv')

NAME_RULES = [
    # Fresh Vegetables
    (r'\b(garlic|onion|ginger|squash|pumpkin|kangkong|tomato|sili|labuyo|sigang|panigang|chopseuy|sweet potato|camote|togabang|bell pepper|finger pepper|spring onion|native tomato|sayote|upo|patola|singkamas|alugbati|saluyot|kamote|ampalaya|sitaw|stringbeans|okra|radish|cucumber|carrots?|potatoes?|leeks?|wansoy|parsley|kinchay|batuan|tanglad|basil|pechay|togue|mushroom|shimeji|broccoli|cauliflower|cabbage|lettuce|spinach|celery|eggplant|corn|mongo|beans|peas|malunggay|bokchoy|sungsong|asparagus|turmeric|chili fingers|greenbell|redbell|sweet potato|togabang)\b', "Fresh - Vegetables"),
    # Fresh Fruits
    (r'\b(mango|banana|apple|grapes?|melon|pineapple|orange|papaya|avocado|lemon|lime|strawberr|watermelon|guava|coconut|calamansi|pomelo|kiwi|lanzones|rambutan|durian|jackfruit|langka|santol|siniguelas|ponkan|guyabano|atis|duhat|balimbing|mangosteen|honeydew|mandarin|navel orange|seedless|red globe|fuji|gala|washington|dragon fruit|lychee|pear|peach|plum|cherry|blueberr|cranberr|raspberry|saba|lakatan|lacatan|guapple|ponkiat|fragrant pears|ya pears|century pears)\b', "Fresh - Fruits"),
    # Fresh Meat
    (r'\b(beef|baka|bulalo|ribeye|rib.?eye|sirloin|tenderloin|wagyu|t.?bone|kalitiran|kenchie|maskara|porterhouse|prime rib|top round|round steak|breakfast steak|korean bbq|chuck rib|ground round|ox tail|ox feet|bone marrow|camto|osso buco)\b', "Fresh - Beef"),
    (r'\b(pork|baboy|liempo|kasim|pigue|belly|chop|lechon|tocino|longganisa|pata|knee cap|ox tripe|lumpiang shanghai|shanghai mix|skinless longanisa|sinigang cut|american ribs|leg quarter|pigs feet|pigs head|adobo cut|jolly pig|cutlet|shank|liver)\b', "Fresh - Pork"),
    (r'\b(chicken|manok|thigh|breast|drumstick|wings?|whole bird|tinola cut|torikatsu|torikaraage|frillers)\b', "Fresh - Chicken"),
    (r'\b(fish|isda|tilapia|bangus|salmon|tuna|shrimp|hipon|squid|pusit|crab|alimango|oyster|tahong|clam|prawn|lapu.?lapu|maya.?maya|bisugo|danggit|dilis|espada|salinas|sapsap|bolinao|salay|lagao|tuyo|hibe|katambak|kasig|tabagak|suahe|pasayan|galunggong|alumahan|marlin|hasa|bolao|bilong|tanguigue|mat.?an|pesogo|crabmeat|crabstick|cuttlefish|fishcake|kingcrab|snowcrab|cod|halibut|bacalao|barramundi|dalagang|kapak|lapad|nokus|dulong|pampano|seabass|mussels|scallop|asuhos|fishball|squidball|alimasag|butcheron|tinapa|dried capak|dried hering|sagana tinapa|san sebastian dried)\b', "Fresh - Seafood"),
    # Frozen
    (r'\b(ice.?cream|gelato|sherbet|popsicle|ice bar|ice dessert|mochi ice|futaba|yawamochi|carmen.?s best|selecta|creamy delight|binggrae|pangtoa|samanco|melona|signature select ic)\b', "Frozen - Ice Cream & Desserts"),
    (r'\b(french fries|crinkle cut|wedge cut|shoestring|hash brown|gyoza|dumpling|mandu|mantou|siomai|xiao long bao|shabu shabu|tteokbokki|bibimbap|fried rice|yakisoba|takoyaki|mochi waffle|corndog|spring roll|allgroo|hana mandoo|k chef|wang.*dumpling|baijia|niko.*gyoza|dorayaki|bambi dumpling|bambi tortilla|bambi pizza|hotsa|ore ida onion|signature select.*fajita|signature select.*edamame|signature select.*pancake|signature select.*waffle|signature select.*pizza|signature select.*battered onion|new senorito.*vegetable|farmland.*vegetable|fat.*thin.*cuttle|fat.*thin.*siopao|marby.*siopao|marby.*lumpia|fried lumpia|kg.*springroll|tsubetsu|uni zritaku|veega)\b', "Frozen - Ready to Heat"),
    (r'\bfrozen\b', "Frozen"),
    # Bakery
    (r'\b(cake|pastry|muffin|croissant|donut|doughnut|ensaymada|pianono|piaya|polvoron|otap|barquillos|hojaldres|rosquillos|biscocho|broas|mamon|cupcake|brownie|waffle|cream puff|cream roll|tart|chiffon|mooncake|tikoy|hopia|pastillas|yema|ube ball|ube bites|taisan|bijon|dinner square|mochi roll|merci.*brojas|merci.*pulceras|margies kinihad|theresas kinihad|de ocampo|eng seng tower|vjandep|navarro.*pastillas|natys best|tian seng|polland ube|shamrock|trevalli|tokyo mochi|fujiya|monteur|hokkaido cream|buko pandan|leche flan|halo.?halo|suman|champorado|ube.*pandan delight|super stix ube|twinkee|snicker.*doodle)\b', "Bakery - Cakes & Pastries"),
    (r'\b(bread|loaf|pandesal|tasty|wheat bread|bun|bagel|baguette|brioche|sourdough|focaccia|pita|tortilla|wrap|roti|monay|kababayan|burger bun|hotdog bun|dinner roll|mini roll|marby buns|village gourmet|gardenia|uncle george|st ives|turks pita|angelina thick|marby.*crust|marby.*tortilla|marby.*ubod|mission wraps|bambi.*taco|bambi.*tortilla|bambi.*molo|lumpia wrapper|wantan wrapper|shao mai wrapper|dae jang gum|jimcu|jcd lumpia|tcvl lumpia)\b', "Bakery - Bread"),
    # Dairy
    (r'\b(butter|margarine|buttercup|dari creme|anchor butter|anchor.*unsalted|anchor.*salted|president unsalted|magnolia buttercup)\b', "Dairy - Butter & Margarine"),
    (r'\b(cheese|quezo|queso|kesong|cheddar|gouda|havarti|parmesan|mozzarella|emmental|brie|feta|quickmelt|quick melt|cheezee|magnolia cheezee|eden cheddar|arla|bega|california cheddar|emborg|sterilgrda|magnolia quickmelt|eden singles|happy valley shredded)\b', "Dairy - Cheese"),
    (r'\b(yogurt|yakult|cultured milk|chobani|dairy farmers|pascual yogtetra|creamy delight greek|milkana|dutchmill|coco gurt|alaska fruitti)\b', "Dairy - Yogurt"),
    (r'\b(fresh eggs|brown eggs|salted eggs|salted duck|elrich.*egg|bounty.*egg|alto palo.*egg|\begg\b|itlog)\b', "Dairy - Eggs"),
    (r'\b(powdered milk|milk powder|growing.?up milk|bear brand|birch tree|anchor.*grain|anchor.*protein|anlene|sustagen|pediasure|ensure|nestogen|bonakid|ascenda|lactum|enfagrow|nido\b|s26\b|promil|nan\b|similac|enfamil|follow.?on|infant formula|fortified powder|adult boost|anmum materna|energen champion)\b', "Dairy - Powdered Milk"),
    (r'\b(fresh milk|full cream|low fat milk|skim milk|uht milk|evaporated|condensada|condensed|all purpose cream|nestle cream|magnolia apc|cooking cream|whipping cream|sour cream|crema|kremdensada|hopla|everchef|emborg sour|bulla|president cooking|president whipping|president skimmed|selecta farm fresh|selecta filled|selecta non fat|jersey|milk magic|vitasoy|vitamilk|nutriboost|chuckie|alaska evaporada|angel all purpose|angel kremdensada|angel creamer|liberty condensada|cream all creamer|nestle coffeemate|blend 45 kondensada|jollycow|nestle lowfat|luv.?lov buddy berry|ev cream whipped|farm fresh cafe latte)\b', "Dairy - Milk & Cream"),
    # Pantry
    (r'\b(cooking oil|vegetable oil|olive oil|canola|palm oil|coconut oil|minola|dona elena|goodlife sesame|sesame oil|mua yu|truffle oil|kadoya|mirin|ryoroushu|cooking sake|tartuffi.*oil|ottogi red pepper oil)\b', "Pantry - Oil"),
    (r'\b(tortellini|gnocchi|rigatoni|linguine|orzo|farfalle|tagliatelle|lasagna|pasta\b|spaghetti|penne|fettuccine|macaroni|vermicelli|miki|hofan|lungkou|somen|couscous|ideal gourmet|gallo|barilla|san remo|sunshine saucy ghetti|sunshine saucey ghetti|mega prime vermicelli|pearl tower|jumy lungkou|tiffany bijon|lucky me pancit|lucky me kasalo|payless.*pancit|quickchow.*pancit|pabo pancit|hapimi pancit|excellent curry pancit|simply chef pancit|fu cheng wu|showa japanese.*noodle|nissin bar noodle|nissin mini noodle|ottogi.*noodle|maggi curry noodle|kanesu cha soba|noodle|pancit|mami|udon|sotanghon|bihon|misua|imee vegetable|nutrifam shirataki|six fortune|sunlee yellow|toasted miki|goodluck toasted|maruchan|tablemark|ramyun|ramen|shin ramyun|nongshim|nissin cup|cup noodle|bowl noodle|samyang|yakisoba|gaga goreng|happy mie|mi goreng|koka instant|mi sedaap)\b', "Pantry - Pasta & Noodles"),
    (r'\b(sardine|mackerel|tuna can|corned beef|luncheon meat|spam\b|555\b|hakone|king cup sardines|ocean flavor|unipak|mega mackerel|master.*sardines|master.*mackerel|reno potted|capri.*pitted|capri.*tomato|fiamma|molinera.*bean|molinera.*anchovy|molinera.*tomato|falcons valley.*corn|sunbest blueberries|jolly.*peach|del monte.*peach|del monte.*spag|ufc tomato catsup|langers cocktail|carne norte|star carne|karne norte|giniling|argentina giniling|sm bonus spanish sardines|reno.*liver|star nutri meats liver|master fried sardines)\b', "Pantry - Canned Goods"),
    (r'\b(jam|jelly|peanut butter|mayo|mayonnaise|ketchup|dressing|ranch|best food|lady.?s choice|sandwich spread|magnolia sandwich|eden sandwich|thousand island|st dalfour|fruit spread|ovomaltine|del monte creamy.*cheesy|heinz.*catsup|heinz.*tomato|cheez whiz)\b', "Pantry - Spreads & Dressing"),
    (r'\b(soup|broth|porridge|congee|lugaw|arroz caldo|miso|camp bells|cream of mushroom|massel|vegetable stock|laksa paste|pho|ultra cubes|por kwan|hai.?s instant)\b', "Pantry - Soups & Porridge"),
    (r'\b(oat|cereal|granola|muesli|cornflakes|kelloggs|quaker|australia harvest|carmans|sante whole grain)\b', "Pantry - Oats & Cereals"),
    (r'\b(stevia|equal|splenda|honey|muscovado|brown sugar|white sugar|sweetener|syrup|mana.*sugar|alter trade|island coco sato|mascobado|iodized salt|rock salt|sunbeam|sm bonus refined sugar|sea salt|white diamond sea salt|coppola sale marino|himalayan.*salt|pink crystal himalayan|sa himalayan|sm bonus.*sugar|sunbeam iodized)\b', "Pantry - Honey & Sweeteners"),
    (r'\b(magic sarap|knorr|ajinomoto|seasoning|marinade|gravy|broth cube|carbonara.*mix|pasta sauce mix|white king|baking mix|cake mix|pancake mix|flour mix|mccormick|lasap mix|s&b golden curry|real thai|ahg.*curry|3 chefs.*curry|way.*sauce|kasugai tempura|tempura batter|caramba|american garden|franks red hot|french mustard|heinz.*mustard|kraft.*mustard|morehouse mustard|frenchs|worcestershire|sweet baby rays|ragu|sig sel barbecue|kraft barbecue|o.food|cj hechandle|cj gochujang|ebara sauce|sig sel sloppy|sig sel.*island|maruha.*yakisoba|kimchi|atsara|gochujang|ssamjang|hoisin sauce|satay sauce|toban sauce|chili garlic|tinapa.*sauce|deep dips|amoy toban|pantai|ottogi red pepper|mothers best chili|hot bee crunchy garlic|wrights liquid smoke|tartuffi.*sauce|yamaki.*bonito|marumiya|surasang seasoned|wang red pepper|old engworchester|way biryani|way paella|way sambal|del monte spag|ufc cheesy|ufc meaty|ufc creamy|ufc oppa|ufc shake|ufc sweet pickle|ufc fun chow|ufc kusina|seaglow|green harvest pickle|ram relish|ram pickles|sig sel artichoke|sig sel sauerkraut|mothers best spiced|lasap breadng|tita lulu.*champorado)\b', "Pantry - Condiments & Sauces"),
    (r'\b(soy sauce|fish sauce|oyster sauce|toyo|patis|bagoong|alamang|hot sauce|lechon sauce|bbq sauce|tomato sauce|tomato paste|chili oil|chili sauce|pesto|sinamak|sukang puti|sukang pula|vinegar|camel sukang|rikoys|marca pina|silver swan|datu puti|lorins coco suka)\b', "Pantry - Condiments & Sauces"),
    (r'\b(rice|bigas|sinandomeng|jasmine rice|dinorado|adlai|quinoa|chia|flaxseed|buckwheat|millet|sorghum|golden phoenix|harvesters|rap and rai malagkit|rap and rai monggo|fc hi fiber|grain fusion|long grain|short grain)\b', "Pantry - Rice & Grains"),
    (r'\b(flour|cornstarch|gawgaw|liwayway|la estrella|sm bonus all purpose flour|baking soda|baking powder|cream of tartar|gelatin|knox gelatin|ferna gelatine|yeast|breading mix|menu breading|amoren crispy coating|ellie all purpose|polar bear tapioca|tapioca pearl|sunshine sago|golden spoon instant barley|camote powder|potato starch|fat.*thin.*cassava|fat.*thin.*superior)\b', "Pantry - Baking"),
    (r'\b(spice|paprika|cinnamon|curry powder|turmeric powder|cayenne|chili powder|cumin|cardamom|coriander|bayleaf|star anise|sesame seed|ispice|badia|mccormick cinnamon|arbis spanish paprika|shien shien paprika|shien shien onion powder|shakti baba|green forest curry|dj spice|sfuw|sanwa wasabi|yamasa tempura|fat.?thin star anise|black pepper|cracked pepper|whole pepper|garlic powder|ginger powder|onion powder|paminta|kasubha|criseldas black pepper|twin m black pepper|mc cormick garlic|arbis fried minced garlic)\b', "Pantry - Spices & Seasonings"),
    # Snacks
    (r'\b(nori|seaweed|wakame|mizuho norimaki|regent crisps seaweed|kangkong chips|shadrachs.*kangkong|cf kangkong|kco kangkong|kangkong king chips)\b', "Snacks - Seaweed & Veggie Chips"),
    (r'\b(marshmallow|markenburg|marken longlegs|beardy monster|beardy great|sweet dart|ring pop|lala ube pastillas|navarro pastimallows|eng seng tower yema|vjandep pastel|frooty milky pop|tiwi golden coin|nelson sweets halloween|christmas lollipop|halloween.*lollipop|halloween.*twirly|halloween mini|striking popping lollipop|rgies butterscotch|merci butterscotch|columbia.?s yakee|choko choko)\b', "Snacks - Candies"),
    (r'\b(candy|gummy|trolli|chewing gum|menthol gum|columbias icool|columbias froot|columbias pintoora|columbias v fresh|mentos|halls|tic.?tac|hi.?chew|skittles|warheads|haribo|cokoc|halloween bears|lush sour|sweet station|gingerbon|fisherman.?s friend|ricola|himalaya berry|himalaya pepermint|himalaya vajomba|chupa chups|lipps pop stix|babble joe tutti|ya yammy tamarind|ya yammy sampalok|wrigley|doublemint|ice breakers mints|mogu mogu|jele cube|jelliyum|wl dip dip|nni nni|piccadelli armada|gandour unica|coco colorful coin|coco energy bar)\b', "Snacks - Candies"),
    (r'\b(biscuit|cracker|cookie|wafer|oreo|skyflakes|fita|rebisco|mcvitie|digestive|hwa tai|haitai|kokola malkist|malkist|juju milk cream|danish milk sandwich|danish strawberry|danish potato biscuits|julie.?s|monde|suncrest|leslie|snacku vegetable crackers|croley|jeanne.*jamie|shapes danish|shapes ugoy|rimi danish|rimi gifts scottisch|cowhead|carmans cookies|carmans greek|lotus biscoff|arnotts timtam|pocky|glico|pringles|ruffles|chips delight|brownie break|m.y. san graham|noceda jacobina|nacho chips|fibisco ginger snaps|want want senbei|lotte kkokalcone|bakers best|orion custard softcake|kokola kukis|kokola milky vanilla|mondelez tiger bites|sunshine golden mix|sunshine premium mix)\b', "Snacks - Biscuits & Crackers"),
    (r'\b(chip|crisp|piattos|nova|tortilla chips|popcorn|chicharon|kropek|lays|doritos|cheetos|oishi|ruffles|bawang na bawang|california cassava|bahaghari taro chips|maxi tropical roots|cravewell purple roots|ya yammy chichacorn|expo coated peanuts|expo greaseless|chickboy popnik|goya take it|maxi tropical|frabelle.*dilis|frabelle.*sweet.*spicy|farmer john potato chips|j&j potato chips|jack.*jill.*potato|jack.*jill mr chips|hunter potato chips|irvins truffle|potato shoe string|piknik potato|w.?l hex potato|s&w piknik|fritolay rold gold|ore ida onion|orells glazed sweet potato|maxi sweet potato|big mouth corniks|edelyns.*chips|cravewell mixedveg)\b', "Snacks - Chips & Dips"),
    (r'\b(nut|peanut|cashew|almond|pistachio|walnut|trail mix|macadamia|pili nuts|sunflower seeds|cha cheer|smart choice.*almond|smart choice.*cashew|smart choice.*pistachio|smart choice.*walnut|smart choice.*pumpkin|smart choice.*sunflower|blue diamond|heritage almonds|heritage walnuts|sunkist pistachios|katliens mixed nuts|jovy crispy pili|rpm glazed pili|rpm roasted pili|rpm garlic pili|daily fix berry|dailyfix|coco everyday mix nuts|coco sunflower|smart choice dried cranberries|boy bawang|jbc happy peanuts|edelyns adobo peanuts|edelyns garlic balitog|walter crunchy salad croutons|w.?l halo.?halo mix nuts|w.?l cornbits)\b', "Snacks - Nuts & Seeds"),
    (r'\b(chocolate|choco|kitkat|snickers|twix|reese|toblerone|ferrero|chocotube|hershey|cadbury|lindt|belgian|van houten|violet crumble|meiji kouka|goya very berries|arcor bon o bon|kinder|dutche|rimi gifts chocolates|culture blends|sheila g|cloud 9|picc adelli|murgerbon|coco macadamia|coco kokokiss|coco cream bar|coco nougat|garuda gery chocolatos|columbia frooty|hueza toffee|orion milk cream|sabroso cacao|sabroso tsokolate|konu cone bites|konu mini krunch|glico pocky|glico cookies|glico strawberry tin|meiji lucky stick|meiji hello panda|hello panda|fujiya milky|monteur belgian|swiss miss|goya take it 4 fingers|milka strawberry|mizuho hokkaido milk cream|ug cheesy ube pandan)\b', "Snacks - Chocolates"),
    (r'\b(tamarind|sampalok|dried mangoes|cebu dried mangoes|phil brand dried|hueza lengua|minco mangosteen|tots barquiron|tots pinasugbo|sugar kiat|fry.*pop|sweet station|nacho chips colored|nacho chips plain|grinnys grain puff|popping boba|tang disney|luv.?lov buddy|sunshine sago big|super stix ube jr)\b', "Snacks - Others"),
    # Beverages
    (r'\b(coffee|nescafe|kopiko|barako|espresso|3.?in.?1|cafe\b|jardin|bottled coffee|iced coffee|starbucks|ucc coffeeblend|ucc roastmaster|great taste|lo.?r essenso|owl kopitiam|mountain brew|chingu cafe|cantata|nestle house blend|goya matcha latte|brewbrite|good day friz)\b', "Beverage - Coffee & Tea"),
    (r'\btea\b|lipton|nestea|green tea|iced tea|herbal tea|twinings|heath.*heather|korean one ginseng', "Beverage - Coffee & Tea"),
    (r'\b(cocoa|milo|ovaltine|choco malt|chuckie|nutriboost)\b', "Beverage - Cocoa & Malt"),
    (r'\b(juice|four seasons|minute maid|c2\b|powerade|gatorade|flavored drink|tampico|ceres|philbrand|del monte tipco|gaya farm|blue cactus|extra joss|smart c|vida c|rite n lite|pocari sweat|monster energy|teazle zero|paldo pororo|tang disney|100%.*fruitjuice|aldi.*ginger brew)\b', "Beverage - Juice & Drinks"),
    (r'\b(softdrink|soda|cola|pepsi|coke|royal|sprite|mountain dew|7.?up|mirinda|a&w rootbeer|lotte milkis)\b', "Beverage - Softdrinks"),
    (r'\b(mineral water|distilled water|spring water|wilkins|absolute|viva water|summit natural|perrier|sip plus himalayan)\b', "Beverage - Water"),
    (r'\b(beer|red horse|san miguel|pale pilsen|heineken|corona|shandy|san mig light)\b', "Beverage - Beer"),
    (r'\b(wine|martini|rose wine|moscato|merlot|cabernet|lambrusco|chamdor|barefoot|riunite|hardys|yellow tail|woomera|carlos light|tuka\b|fundador|gsm blue|tanduay|rum\b|gin\b|vodka|tequila|liquor|brandy|whisky|whiskey|scotch|bourbon|moutai|dalmore|singleton|macallan|johnnie walker|alfonso|sake\b|hana akita|kome ichizu|ube cream liqueur|walsh curacao|charles.*james)\b', "Beverage - Alcoholic"),
    # Health & Medicine
    (r'\b(capsule|tablet|syrup|supplement|vitamin|mineral|atc\b|pharex|ceelin|centrum kid|enervon|ferrous sulfate|solmux|decolgen|alaxan|flanax|vicks|katinko|salonpas|tiger balm|tigerbalm|ammeltz|rhea cold rub|efficascent|nin jiom|strepsils|bye bye fever|tempra cool|mediplast|polident|dental b tbrush|sansflou|sansfluo|yamang bukid turmeric|white flower eucalyptus|apollo sebo de macho|betet medicated|tai chi.*rub|tai chi.*liniment|tai chi.*menthol|mentopas medicated|twinluck.*face mask|aljohn.*facemask)\b', "Health & Medicine"),
    (r'\b(ethyl alcohol|isopropyl alcohol|biogenic|greencross|safe more|guardian isoprophyl|dr j ethyl|dr j iso|alcoplus|cleene hand sanitizer|bench alcogel|naturelab.*alco|naturelab.*alcohol)\b', "Health & Medicine"),
    # Babies & Kids
    (r'\b(diaper|pampers|huggies|drypers|lampein pants|mamy poko|eq pants)\b', "Babies & Kids - Diapers"),
    (r'\b(nan\b|similac|enfamil|s-26|s26\b|promil|nido\b|follow.?on|infant formula|bonakid|ascenda|lactum|enfagrow|nestogen|pediasure|ensure gold)\b', "Babies & Kids - Formula Milk"),
    (r'\b(baby food|gerber|cerelac|nestum|heinz.*custard|heinz.*vanilla|aveeno baby|johnson baby|babyflo bath|babyflo powder|babyflo cologne|babyflo baby|tiny buds|smart steps baby|lactacyd baby)\b', "Babies & Kids - Baby Food & Care"),
    # Wipes
    (r'\b(baby wipes|cherub baby|kleenex pure water|babyflo cotton buds|tender love wipes|nurture.*wipes|nursy wipes|uni love wipes|unilove.*wipes|punaas.*wipes|best lab.*wipes|bestlab.*wipes|bestlab antibacterial|best lab antibacterial|best lab mini wipes|sanicare.*wipes|cleene cotton|cuddles cotton|megan cotton|wipe cotton balls|wipe absorbent cotton|sm bonus cotton absorbent|sm bonus pet wipes|sm bonus disposable gloves|unicare cleansing wipes|uni love soft unscented)\b', "Home Care - Paper & Disposables"),
    # Home Care
    (r'\b(ariel|tide\b|surf\b|downy|fabric conditioner|zonrox|bleach|detergent|laundry|breeze capsules|breeze powder|speed det|calla det|speed bar|kalinisan fabcon|snuggle fabcon|uni love fabric|unilove peppa)\b', "Home Care - Laundry"),
    (r'\b(lysol|ajax|domex|mr\.? clean|pinesol|toilet bowl|floor wax|cleaner|krest disinfectant|family guard disinfectant|mighty mom dishwashing|joy dishwashing|arm.*hammer|mr muscle|febreze|febreeze|natucair|farcent|ambi pur|glade|koala tablet deodorizer|easy scoop odor|vet core odor|porma shoe|cleans up easy grip|sm bonus scrubbing pad|best lab handsoap|energizer max|energizer.*battery)\b', "Home Care - Cleaning & Air Care"),
    (r'\b(baygon|raid|insect|mosquito|cockroach|pest|hoyhoy trap|advanced tracking powder|bayopet flea|shield gard flea|doggies choice tick|petgard dog powder)\b', "Home Care - Pest Control"),
    (r'\b(tissue|paper towel|napkin|trash bag|garbage bag|sando bag|cling wrap|aluminum foil|cotton roll|cotton buds|cotton pads|ziploc|freezer bag|disposable cups|straw\b|wooden chopstick|wooden stirrer|paper cup|paper meal box|hamburger box|sauce cup|bento box|lil princess|little princess|carnival.*cups|glow.*bag|glow.*tray|glow.*zip bag|glow.*paper cups|glow.*toothpick|glow ice bag|glow bio fork|glow bio spoon|rollo bio|calypso options|basong pinoy|happy bee aluminum|cheers aluminum|cheers baking paper|panbake|starchware|eco innovators|uncle johns|skz bento|bouncy vp|bouncy kitchen towel|happy cotton rolls|wipe absorbent|femme facial.*travel|sm bonus kitchen towel|tn pre cut|sm bonus paper meal box|sm bonus hamburger box|sm bonus biodegradble)\b', "Home Care - Paper & Disposables"),
    (r'\b(candle|esperma|manila wax|liwanag)\b', "Home Care - Others"),
    # Personal Care
    (r'\b(shampoo|pantene|sunsilk|rejoice|head.?shoulders|dandruff|creamsilk|being conditioner|mane n tail|tresemme|kerasys|tsubaki|keratin plus|vitakeratin|pregroe|symply g.*cond|symply g.*treatment|l.?oreal ever pure|black beauty conditioner|hana conditioner|cream silk hair|ellips hair|grips strong hold)\b', "Personal Care - Hair"),
    (r'\b(hair treatment|hair dye|hair color|hair colorant|garnier hair|loreal exclusive|icolor|megan hair color|kolours hair dye|vitress|grips hair|gatsby.*wax|gatsby.*pomade|gatsby.*gel|gatsby.*spray|bench fix|hair clay|hair gel|hair wax|hair pomade|hair spray)\b', "Personal Care - Hair"),
    (r'\b(deodorant|roll.?on|antiperspirant|body spray|body wash|shower gel|safeguard|palmolive|dove\b|nivea|vaseline|lotion|prickly heat|fissan|talcum powder|enchanteur powder|deoplus|old spice|axe\b|gatsby body spray|gatsby cologne|bench body spray|bench daily|blackwater deo|black water deo|bw women deo|penshoppe deo|penshoppe body|penshoppe bodyspray|penshoppe pocket scents|secret deo|gillette deo|brut deo|cathydoll deo|active white salt scrub|active white moisturizer|hello glow lot|block.*glow lot|ar body cream|ar moisturizing|ar whitening|gt moisturizing|gt whitening|kawaii max gluta|kojie san sun protect|gluta body soap|chupa chups bodywash|aveeno bodywash|hello glow sunflower|hello glow 2in1|hello glow hair removal|lucas papaw|mei yi massage|apollo sebo de macho)\b', "Personal Care - Body"),
    (r'\b(soap\b|bioderm|hygienix|jergens|shield soap|perla\b|activex|tokyo white body soap|seaoul white body soap|seoul white.*soap|gluta.*soap|kojie san.*soap|kojiesan|active white.*soap|ar soap|beauche.*soap|dr alvin kojic soap|dr wong|bevi body soap|nature glutathione|kawaii max glutathion|symply g kojic soap|vet core active soap|defensil|jb soap|rexona)\b', "Personal Care - Body"),
    (r'\b(toothpaste|toothbrush|mouthwash|colgate|closeup|oral.?b|listerine|sansflou|sansfluo|polident|dentiste)\b', "Personal Care - Oral"),
    (r'\b(facial wash|toner|sunscreen|face cream|serum|olay|cetaphil|neutrogena|pond\b|clean.?clear|garnier.*facial|garnier.*men|garnier.*essence|garnier light|gatsby facial|gatsby.*scrub|senka|celeteque|skin white|iwhite|belo\b|hello glow nose|hello glow lip|beauty formulas|k.?glow ice scrub|gt bleaching|ever bilena|maybelline|nail polish|nail tips|fake nail|omg.*nail|bobbie nail|bobbie.*cuticle|bobbie.*top coat|my nails nail|watsup50 nail|maxi beauty astringent|cathydoll facial|cathydoll ffoam|dermplus|careline.*bb cream|careline.*brow|careline.*liner|careline.*mascara|careline.*powder|careline.*concealer|careline.*tint|careline.*eyebro|myra facial|beauche facial)\b', "Personal Care - Facial"),
    (r'\b(feminine wash|feminine spray|lactacyd feminine|ph care|gluta c feminine|naflora|betadine antiseptic feminine|hers feminine wipes|charmee menstrual|sisters.*menstrual|sisters.*sanitary|modess|sofy|secure men pads)\b', "Personal Care - Feminine"),
    (r'\b(cologne|perfume|body mist|ellips cologne|juicy cologne|bench daily scent|nenuco|naturelab cologne|naturelab bt21 fragrance|natuelab.*fragrance|herbench|babyflo cologne|penshoppe pocket scents)\b', "Personal Care - Fragrance"),
    # Pet Care
    (r'\b(dog food|cat food|pedigree|whiskas|purina|alpo|minino|goodest catfood|moochie|nutripet|doggo|lucky dog|dentalight|pet plus dental|top2tail|topdog|playpets|doggies choice|petgard|our cat|easy scoop cat|bayopet|woof n tail|kg pet round|calming bed|pet bed|foam bag ripstop|shield gard flea and tick|dog bowl)\b', "Pet Care"),
    # Ready to Cook / Processed Meats
    (r'\b(ham\b|bacon|jamon|chorizo|holiday ham|hamonado|hotdog|sausage\b|franks|longganisa|hungarian sausage|salami|pepperoni|bratwurst|schublig|cabanossi|debreziner|nuremberg|vienna sausage|meatloaf|meatball|frankfurter|cdo idol|cdo jamon|cdo karne|cdo crispy|cdo ulam|cdo holiday|cdo premium|star nm sausage|star nm ulam|virginia|winner hamonado|winner hotdog|purefoods tender juicy|purefoods fs|bounty fresh|fat.*thin.*sausage|fat.*thin.*longanisa|kwong bee sausage|black bridge sausage|sajo cocktail|stefan|king sue|belcris|bellshayce|aguila gourmet|aguila chorizo|aguila hungarian|el rancho burger|purefoods.*burger|food service.*patty|food service.*corndog|tapa king|tulip jamonilla|tulip natural|j&f.*tocinitos|fantastic sausa|magnolia.*frillers|mm adobo cut|meat master adobo|jolly pig adobo|pure foods rte adobo|cdo danes)\b', "Ready to Cook - Processed Meats"),
    (r'\b(ulam\b|humba|kare.?kare|caldereta|menudo|mechado|afritada|pinakbet|paksiw|bistek|bicols best|goldilocks laing|ks specialty sisig|max.?s crispy pata|tapa\b)\b', "Ready to Cook - Filipino"),
    (r'\brtc\b|ready.?to.?cook|marinated|breaded', "Ready to Cook - Marinated"),
    # Non-Food
    (r'\b(kitchenware|dispenser|container|storage|utensil|plate\b|mug\b|spatula|ladle\b|tong\b|watts houseware|watts accessories|watts baking|watts stationery|watts diy|watsup50|creazions shoe cabinet|astron blender|astron flat iron|astron multi pot|hanabishi blender|eureka dry flat iron|eureka microwave|tough mama|toughmama|tenki cables|tenki razor|ramgo microgreens|ramgo packet|ramgo rosemary|ramgo sprout|aster chair|pumpkin pop|christmas tree tops|christmas snowman|watts car sponge|eurochef carving knife)\b', "Non-Food - Household"),
    # Gift Sets
    (r'\b(fruit basket|gift set|hamper|promo pack|bundle|assorted fruits 13 kinds|eden basket|sm bonus basket|mnv fruit|sm bonus j basket)\b', "Promos & Gift Sets"),
]

def assign_category(name):
    for pattern, category in NAME_RULES:
        if re.search(pattern, str(name), re.IGNORECASE):
            return category
    return 'Uncategorized'

df['item_category'] = df['item_name'].apply(assign_category)
still = (df['item_category'] == 'Uncategorized').sum()
print(f'Still uncategorized: {still} / {len(df)}')
print(f'Unique categories: {df["item_category"].nunique()}')
df.to_csv('results/grocery_data.csv', index=False, encoding='utf-8-sig')
print('Saved!')

Still uncategorized: 373 / 10052
Unique categories: 63
Saved!
